# NB41 — Sample-Level CWM OLS: pH Mediation Test

**Motivation:** NB40 showed that genus-level PGLS (Pagel's λ ≈ 0.9) cannot confirm or refute
mediation of the pH → metal gene density signal because no positive control is achievable within
the PGLS framework. This notebook tests mediation at the **sample level** using OLS on
community-weighted mean (CWM) gene density, which is free of the phylogenetic signal constraint.

**CWM:** `CWM_ko[sample] = Σ_genus(RA_genus × ko_per_mb_genus) / Σ_genus(RA_genus)`
This was precomputed via Spark (`scripts/h3a_cwm_analysis.py`, coverage ≥ 0.05 filter)
and stored in `data/h3a_cwm_sample_data.csv` (83,401 unique samples; pH non-null: 64,466).

**Design:**
- Section A: Baseline `cwm_ko ~ pH_z` — establish direction (compare to H3a result)
- Section B: SOM mediation `cwm_ko ~ pH_z + SOM_z` — positive control (SOM was tested but
  showed Δβ ≈ 0% in PGLS; OLS is the correct framework to test this)
- Section C: GeoROC total metal mediation — Zn alone and PC1 of all 6 GeoROC metals
- Section D: Summary forest plot comparing β_pH across mediation levels
- Section E: CWM_PF1 scaffold (Spark-only; requires genus_ra.parquet on-cluster)

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H, grid_h
apply_style()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

DATA = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data')
FIGS = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/figures')

try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    _SPARK_AVAILABLE = True
    print('Spark available:', spark.version)
except Exception as e:
    _SPARK_AVAILABLE = False
    print(f'Spark unavailable: {e}')

def sig(p):
    if p is None: return ''
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

Spark available: 4.0.1


In [2]:
# Load h3a CWM data and env extended
cwm_raw = pd.read_csv(DATA / 'h3a_cwm_sample_data.csv')
cwm_raw = cwm_raw.drop_duplicates(subset='sample_id', keep='first')
env = pd.read_csv(DATA / 'cwm_sample_env_extended.csv')

# Merge env covariates
df = cwm_raw.merge(env, on='sample_id', how='inner')

# Base filter (matches h3a_cwm_analysis.py filter)
df = df[(df['cwm_ko'] > 0) & (df['coverage'] >= 0.05)]
print(f'After cwm_ko>0 + coverage>=0.05: {len(df):,} samples')
print(f'soil_pH non-null: {df["soil_pH"].notna().sum():,}')
print(f'soil_som_pct non-null: {df["soil_som_pct"].notna().sum():,}')

geo_cols = ['georoc_Cu', 'georoc_Ni', 'georoc_Zn', 'georoc_Co', 'georoc_Cr', 'georoc_Pb']
for col in geo_cols:
    print(f'  {col} non-null: {df[col].notna().sum():,}')

# Z-score predictors on the full pH-valid set
df_ph = df.dropna(subset=['soil_pH']).copy()
print(f'\nAnalysis base (pH non-null): {len(df_ph):,}')

def z(s): return (s - s.mean()) / s.std()

df_ph['pH_z'] = z(df_ph['soil_pH'])

After cwm_ko>0 + coverage>=0.05: 83,401 samples
soil_pH non-null: 64,466
soil_som_pct non-null: 64,466
  georoc_Cu non-null: 16,161
  georoc_Ni non-null: 23,688
  georoc_Zn non-null: 16,975
  georoc_Co non-null: 17,715
  georoc_Cr non-null: 22,842
  georoc_Pb non-null: 19,316

Analysis base (pH non-null): 64,466


## Section A — Baseline: `cwm_ko ~ pH_z`

Replicate the H3a direction check at sample level. Expected: negative β_pH
(high-pH communities carry less metal gene mass per Mb).

In [3]:
r_base = smf.ols('cwm_ko ~ pH_z', data=df_ph).fit()
b_pH_base = r_base.params['pH_z']
p_pH_base = r_base.pvalues['pH_z']
ci_base = r_base.conf_int().loc['pH_z']
r2_base = r_base.rsquared

print(f'OLS cwm_ko ~ pH_z')
print(f'  β_pH = {b_pH_base:+.4f}  95% CI [{ci_base[0]:+.4f}, {ci_base[1]:+.4f}]  {sig(p_pH_base)}')
print(f'  p    = {p_pH_base:.2e}')
print(f'  R²   = {r2_base:.4f}')
print(f'  n    = {r_base.nobs:.0f}')

OLS cwm_ko ~ pH_z
  β_pH = +0.2442  95% CI [+0.2235, +0.2649]  ***
  p    = 1.53e-117
  R²   = 0.0082
  n    = 64466


## Section B — SOM Mediation: `cwm_ko ~ pH_z + SOM_z`

SOM showed Δβ ≈ 0% in genus-level PGLS (NB40 Section C), but PGLS with λ ≈ 0.9 is resistant
to covariate attenuation. OLS is the appropriate positive-control framework.

**Note on SOM and pH coverage:** Both `soil_pH` and `soil_som_pct` are sourced from the same
global soil data layer (SoilGrids/OpenLandMap), so their non-null counts are expected to match
(n ≈ 64,466). If Δβ > 20%, SOM mediates pH → cwm_ko; if Δβ < 20%, pH is independent of SOM
at the sample level — and the PGLS null result is confirmed by OLS.

**Fe, Mn note:** Fe and Mn are sorbent matrix elements (Fe/Mn oxides bind heavy metals in soil),
not target metals. They are not included in the CSU PF1 mobile fraction dataset, and GeoROC does
not include Fe or Mn. NGSA has Fe/Mn ICP-MS but only for Australian samples (Spark-only). Not
included here.

In [4]:
# SOM mediation
df_som = df_ph.dropna(subset=['soil_som_pct']).copy()
df_som['SOM_z'] = z(df_som['soil_som_pct'])

# Baseline on SOM subset (for proper Δβ comparison)
r_base_s = smf.ols('cwm_ko ~ pH_z', data=df_som).fit()
b_pH_s0 = r_base_s.params['pH_z']

# + SOM
r_som = smf.ols('cwm_ko ~ pH_z + SOM_z', data=df_som).fit()
b_pH_som = r_som.params['pH_z']
b_SOM = r_som.params['SOM_z']
p_pH_som = r_som.pvalues['pH_z']
p_SOM = r_som.pvalues['SOM_z']
ci_som = r_som.conf_int().loc['pH_z']
delta_som = (b_pH_s0 - b_pH_som) / abs(b_pH_s0) * 100

print(f'SOM mediation test (n = {len(df_som):,})')
print(f'  Baseline β_pH        = {b_pH_s0:+.4f}')
print(f'  +SOM β_pH            = {b_pH_som:+.4f}  {sig(p_pH_som)}')
print(f'  β_SOM                = {b_SOM:+.4f}  {sig(p_SOM)}')
print(f'  Δβ_pH                = {delta_som:+.1f}%')
verdict = 'MEDIATED (Δβ > 20%)' if delta_som > 20 else 'ROBUST (Δβ ≤ 20%)'
print(f'  Verdict              : {verdict}')
print(f'  R²                   = {r_som.rsquared:.4f}')
print()
print('Comparison with NB40 PGLS SOM result: Δβ ≈ 0% (λ≈0.9 framework-resistant)')

SOM mediation test (n = 64,466)
  Baseline β_pH        = +0.2442
  +SOM β_pH            = +0.2375  ***
  β_SOM                = -0.0151  ns
  Δβ_pH                = +2.8%
  Verdict              : ROBUST (Δβ ≤ 20%)
  R²                   = 0.0082

Comparison with NB40 PGLS SOM result: Δβ ≈ 0% (λ≈0.9 framework-resistant)


## Section C — GeoROC Metal Mediation: Zn and composite PC1

GeoROC measures long-term crustal total metal concentrations (parent bedrock geochemistry).
This is NOT the same as bioavailable fraction (CSU PF1 tested in NB40). Testing both provides
complementary evidence: if neither total nor bioavailable metal concentrations mediate pH → cwm_ko,
the pH effect is likely acting through direct physiology or community assembly rather than
metal-specific selection pressure.

**GeoROC coverage:** Only ~16,975 of 83,401 samples have GeoROC Zn data (bedrock geochemistry
grid coverage is patchy). All 6 metals are available for ~14k samples. The n-reduction may bias
results; we report Δβ within each subset's own baseline.

In [5]:
# --- GeoROC Zn alone ---
df_zn = df_ph.dropna(subset=['georoc_Zn']).copy()
df_zn['georoc_Zn_z'] = z(df_zn['georoc_Zn'])

r_base_zn = smf.ols('cwm_ko ~ pH_z', data=df_zn).fit()
b_pH_zn0 = r_base_zn.params['pH_z']

r_zn = smf.ols('cwm_ko ~ pH_z + georoc_Zn_z', data=df_zn).fit()
b_pH_zn = r_zn.params['pH_z']
b_Zn = r_zn.params['georoc_Zn_z']
p_pH_zn = r_zn.pvalues['pH_z']
p_Zn = r_zn.pvalues['georoc_Zn_z']
ci_zn = r_zn.conf_int().loc['pH_z']
delta_zn = (b_pH_zn0 - b_pH_zn) / abs(b_pH_zn0) * 100

print(f'GeoROC Zn mediation (n = {len(df_zn):,})')
print(f'  Baseline β_pH = {b_pH_zn0:+.4f}')
print(f'  +Zn β_pH      = {b_pH_zn:+.4f}  {sig(p_pH_zn)}')
print(f'  β_Zn          = {b_Zn:+.4f}  {sig(p_Zn)}')
print(f'  Δβ_pH         = {delta_zn:+.1f}%')
print()

# --- GeoROC composite PC1 (all 6 metals) ---
df_geo = df_ph.dropna(subset=geo_cols).copy()
X_geo = StandardScaler().fit_transform(df_geo[geo_cols].values)
df_geo = df_geo.copy()
df_geo['georoc_PC1_z'] = PCA(n_components=1).fit_transform(X_geo)

r_base_geo = smf.ols('cwm_ko ~ pH_z', data=df_geo).fit()
b_pH_geo0 = r_base_geo.params['pH_z']

r_geo = smf.ols('cwm_ko ~ pH_z + georoc_PC1_z', data=df_geo).fit()
b_pH_geo = r_geo.params['pH_z']
b_PC1 = r_geo.params['georoc_PC1_z']
p_pH_geo = r_geo.pvalues['pH_z']
p_PC1 = r_geo.pvalues['georoc_PC1_z']
ci_geo = r_geo.conf_int().loc['pH_z']
delta_geo = (b_pH_geo0 - b_pH_geo) / abs(b_pH_geo0) * 100

print(f'GeoROC PC1 (6-metal composite) mediation (n = {len(df_geo):,})')
print(f'  Baseline β_pH = {b_pH_geo0:+.4f}')
print(f'  +PC1 β_pH     = {b_pH_geo:+.4f}  {sig(p_pH_geo)}')
print(f'  β_PC1         = {b_PC1:+.4f}  {sig(p_PC1)}')
print(f'  Δβ_pH         = {delta_geo:+.1f}%')
print()
print('Note: GeoROC = total crustal chemistry (bedrock); CSU PF1 (NB40) = bioavailable fraction')
print('Neither mediates the pH → cwm_ko signal if both show Δβ < 20%')

GeoROC Zn mediation (n = 14,817)
  Baseline β_pH = +0.0806
  +Zn β_pH      = -0.0057  ns
  β_Zn          = -0.3749  ***
  Δβ_pH         = +107.0%

GeoROC PC1 (6-metal composite) mediation (n = 7,299)
  Baseline β_pH = -0.1167
  +PC1 β_pH     = -0.1170  ***
  β_PC1         = -0.0018  ns
  Δβ_pH         = +0.2%

Note: GeoROC = total crustal chemistry (bedrock); CSU PF1 (NB40) = bioavailable fraction
Neither mediates the pH → cwm_ko signal if both show Δβ < 20%


## Section D — Summary Figure: β_pH across mediation models

Forest plot showing β_pH (95% CI) at each control level. Right panel shows % attenuation (Δβ)
with comparison to NB40 PGLS null result.

In [6]:
# Compile model results
models = [
    ('Baseline', b_pH_base,
     r_base.conf_int().loc['pH_z', 0],
     r_base.conf_int().loc['pH_z', 1],
     p_pH_base, 0.0, int(r_base.nobs), 'OLS'),
    ('+SOM', b_pH_som,
     r_som.conf_int().loc['pH_z', 0],
     r_som.conf_int().loc['pH_z', 1],
     p_pH_som, delta_som, len(df_som), 'OLS'),
    ('+GeoROC Zn', b_pH_zn,
     r_zn.conf_int().loc['pH_z', 0],
     r_zn.conf_int().loc['pH_z', 1],
     p_pH_zn, delta_zn, len(df_zn), 'OLS'),
    ('+GeoROC PC1', b_pH_geo,
     r_geo.conf_int().loc['pH_z', 0],
     r_geo.conf_int().loc['pH_z', 1],
     p_pH_geo, delta_geo, len(df_geo), 'OLS'),
]

labels = [m[0] for m in models]
betas  = [m[1] for m in models]
ci_lo  = [m[2] for m in models]
ci_hi  = [m[3] for m in models]
pvals  = [m[4] for m in models]
deltas = [m[5] for m in models]
ns     = [m[6] for m in models]

ys = np.arange(len(models))[::-1]  # top-to-bottom

fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Left: β_pH with 95% CI
ax = axes[0]
ax.axvline(0, color='gray', lw=0.8, ls='--')
for i, y in enumerate(ys):
    color = PALETTE[1] if models[i][7] == 'OLS' else '#888888'
    ax.errorbar(betas[i], y,
                xerr=[[betas[i] - ci_lo[i]], [ci_hi[i] - betas[i]]],
                fmt='o', color=color, ecolor=color, capsize=3, ms=5, lw=1.2)
    ax.text(betas[i], y + 0.25, sig(pvals[i]), ha='center', va='bottom', fontsize=7)
    ax.text(max(ci_hi) + 0.005, y, f'n={ns[i]:,}', ha='left', va='center',
            fontsize=7, color='#808080')

ax.set_yticks(ys)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('β_pH (CWM_ko per SD pH_z)')
ax.set_ylabel('Model')
ax.set_title('pH coefficient by OLS model', fontsize=10)
grid_h(ax)

# Right: % attenuation
ax2 = axes[1]
ax2.axvline(0, color='gray', lw=0.8, ls='--')
ax2.axvline(20, color='darkred', lw=0.7, ls=':', alpha=0.6, label='20% threshold')
bar_colors = [PALETTE[1] if models[i][7] == 'OLS' else '#cccccc' for i in range(len(models))]
ax2.barh(ys, deltas, color=bar_colors, edgecolor='k', linewidth=0.5, alpha=0.85, height=0.5)
ax2.set_yticks(ys)
ax2.set_yticklabels(labels, fontsize=9)
ax2.set_xlabel('Δβ_pH (%)')
ax2.set_ylabel('')
ax2.set_title('pH signal attenuation (Δβ %)', fontsize=10)
ax2.legend(fontsize=7, loc='lower right')
ax2.annotate('NB40 PGLS range: Δβ < 1%\n(all 6 CSU PF1 metals)',
             xy=(0.02, 0.05), xycoords='axes fraction', ha='left', va='bottom',
             fontsize=6, color='#888888', style='italic',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f5', alpha=0.8))
grid_h(ax2)

fig.suptitle('Sample-level OLS: pH → CWM metal gene density — mediation tests',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
save(fig, FIGS / 'fig_nb41_cwm_ols_mediation')
print('Saved fig_nb41_cwm_ols_mediation.pdf')

Saved fig_nb41_cwm_ols_mediation.pdf


## Section D — Interpretation

**Key comparison point:**
- Genus-level PGLS (NB40): SOM Δβ ≈ 0%, CSU PF1 metals Δβ < 1% — null mediation, but no
  positive control achievable (λ ≈ 0.9 framework resistance)
- Sample-level OLS (this notebook): SOM is the positive control. If SOM shows Δβ > 20% here
  but < 20% in PGLS, it means PGLS was masking real attenuation. If SOM also shows Δβ < 20%
  in OLS, the null mediation finding is confirmed across both frameworks.

**GeoROC total metals vs CSU PF1 bioavailable:**
GeoROC measures long-term parent material geochemistry (total crustal concentrations). CSU PF1
measures the mobile/bioavailable fraction of metals in actual soil samples. Both being non-mediators
would mean pH → metal gene density is not explained by either the amount or the bioavailability
of metals in the environment where these organisms live.

## Section E — CWM_PF1 per Sample (Spark-only scaffold)

Computing per-sample CWM of CSU PF1 bioavailable metal fractions requires genus-level relative
abundance per sample (`genus_ra.parquet`), which is not available locally (off-cluster).

**Formula:** `CWM_PF1_As[sample] = Σ_genus(RA_genus × PF1_As_genus) / Σ_genus(RA_genus)`
where `PF1_As_genus` comes from `data/genus_csu_mobility.csv` (per-genus mean PF1 computed in NB40).

**On-cluster execution:** Run via JupyterHub with Spark session. The code below will execute
when `_SPARK_AVAILABLE = True` and the `genus_ra.parquet` is accessible.

In [7]:
_GENUS_RA_PATH = DATA / 'genus_ra.parquet'
if not _SPARK_AVAILABLE or not _GENUS_RA_PATH.exists():
    print('Section E requires on-cluster execution with genus_ra.parquet available.')
    print(f'  genus_ra.parquet found: {_GENUS_RA_PATH.exists()}')
    print(f'  Spark available: {_SPARK_AVAILABLE}')
    print('Run this notebook in JupyterHub with genus_ra.parquet accessible to compute CWM_PF1 per sample.')
else:
    # Load PF1 per genus (computed in NB40)
    pf1 = pd.read_parquet(DATA / '40_genus_csu_pf1_means.parquet')
    pf1['genus_lower'] = pf1['genus_lower'].str.lower()
    pf1_mean_cols = [c for c in pf1.columns if c.endswith('_mean') and 'n_' not in c]
    print(f'PF1 mean columns: {pf1_mean_cols}')

    # Broadcast PF1 lookup to Spark
    from pyspark.sql import functions as F
    pf1_sdf = spark.createDataFrame(pf1[['genus_lower'] + pf1_mean_cols])

    # Load genus × sample RA (expects: sample_id, genus_lower, relative_abundance)
    ra_sdf = spark.read.parquet(str(_GENUS_RA_PATH))

    # Join to PF1
    ra_pf1 = ra_sdf.join(pf1_sdf, on='genus_lower', how='inner')

    # CWM per sample = weighted mean
    agg_exprs = [
        (F.sum(F.col('relative_abundance') * F.col(c)) /
         F.sum(F.when(F.col(c).isNotNull(), F.col('relative_abundance')))).alias(f'cwm_{c}')
        for c in pf1_mean_cols
    ]
    cwm_pf1_sdf = ra_pf1.groupBy('sample_id').agg(*agg_exprs)
    cwm_pf1_pd = cwm_pf1_sdf.toPandas()
    cwm_pf1_pd.attrs = {}
    out_path = DATA / '41_cwm_pf1_sample_level.parquet'
    cwm_pf1_pd.to_parquet(out_path, index=False)
    print(f'Saved: {out_path}  ({len(cwm_pf1_pd):,} samples)')

Section E requires on-cluster execution with genus_ra.parquet available.
  genus_ra.parquet found: False
  Spark available: True
Run this notebook in JupyterHub with genus_ra.parquet accessible to compute CWM_PF1 per sample.
